# 🧪 Practice Lab: Ensemble Methods

**Netsetos GenAI Course** | Module 0.5 · Lesson 1.7 | ⏱ ~60-90 min

**Prerequisites:** Python 3.10+, numpy, sklearn (pre-installed in Colab)

### 🎯 You will:
1. Train RF and observe variance reduction with more trees
2. Tune GradientBoosting across lr × depth grid
3. Compare hard vs soft voting
4. Build stacking classifier, inspect meta-learner weights
5. Simulate LLM routing with cost-aware cascade
6. Full ensemble pipeline with diagnostic report

---

## Setup — Shared Dataset

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import numpy as np

X, y = make_classification(n_samples=2000, n_features=20,
                           n_informative=12, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_tr.shape}, Test: {X_te.shape}')

---
## Exercise 1 (Easy): Random Forest Bagging

**📖 Concept:** More trees → lower variance. OOB score = free validation.

**📝 Task:** n_estimators = 1, 10, 50, 100, 500. OOB score. Top-5 features.

In [ ]:
# ✏️ YOUR CODE HERE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# TODO: n_estimators sweep, OOB, features

<details><summary>✅ Solution</summary>



In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

for n in [1, 10, 50, 100, 500]:
    rf = RandomForestClassifier(n, random_state=42).fit(X_tr, y_tr)
    print(f'  n={n:>3}: {accuracy_score(y_te, rf.predict(X_te)):.4f}')

rf = RandomForestClassifier(n_estimators=500, oob_score=True, random_state=42).fit(X_tr, y_tr)
print(f'\nOOB: {rf.oob_score_:.4f}')
top5 = rf.feature_importances_.argsort()[-5:][::-1]
print(f'Top-5: {top5}, importances: {rf.feature_importances_[top5].round(3)}')

</details>

---

## Exercise 2 (Easy): Boosting Shootout

**📖 Concept:** lr × depth grid. Lower lr + more trees = better but slower.

**📝 Task:** lr = 0.01, 0.05, 0.1, 0.3 × depth = 2, 3, 5. Find best combo.

In [ ]:
# ✏️ YOUR CODE HERE
from sklearn.ensemble import GradientBoostingClassifier
import time

# TODO: lr × depth grid, find best

<details><summary>✅ Solution</summary>



In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
import time

best_acc, best_cfg = 0, None
for lr in [0.01, 0.05, 0.1, 0.3]:
    for d in [2, 3, 5]:
        t0 = time.time()
        gb = GradientBoostingClassifier(n_estimators=200, learning_rate=lr, max_depth=d, random_state=42).fit(X_tr, y_tr)
        acc = accuracy_score(y_te, gb.predict(X_te))
        m = ' ← best' if acc > best_acc else ''
        if acc > best_acc: best_acc, best_cfg = acc, (lr, d)
        print(f'  lr={lr:>4} d={d}: {acc:.4f} ({time.time()-t0:.1f}s){m}')
print(f'\nBest: lr={best_cfg[0]}, depth={best_cfg[1]}')

</details>

---

## Exercise 3 (Medium): Voting — Hard vs Soft

**📖 Concept:** Hard = majority label. Soft = averaged probabilities.

**📝 Task:** 3 diverse models → individual accuracy → hard → soft. Show soft > hard.

In [ ]:
# ✏️ YOUR CODE HERE
from sklearn.ensemble import VotingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

# TODO: individual, hard voting, soft voting

<details><summary>✅ Solution</summary>



In [ ]:
from sklearn.ensemble import VotingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(n_estimators=100, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
lr = LogisticRegression(max_iter=1000, random_state=42)
est = [('rf', rf), ('gb', gb), ('lr', lr)]

for name, m in est:
    c = m.__class__(**m.get_params()).fit(X_tr, y_tr)
    print(f'  {name:>3}: {accuracy_score(y_te, c.predict(X_te)):.4f}')

for v in ['hard', 'soft']:
    vc = VotingClassifier(estimators=est, voting=v).fit(X_tr, y_tr)
    print(f'  {v:>4} voting: {accuracy_score(y_te, vc.predict(X_te)):.4f}')

</details>

---

## Exercise 4 (Medium): Stacking with Meta-Learner

**📖 Concept:** Meta-learner on OOF predictions. Inspect coef_ for trust weights.

**📝 Task:** StackingClassifier cv=5. Compare all methods. Inspect weights.

In [ ]:
# ✏️ YOUR CODE HERE
from sklearn.ensemble import StackingClassifier

# TODO: stacking, compare, inspect weights

<details><summary>✅ Solution</summary>



In [ ]:
from sklearn.ensemble import (StackingClassifier, VotingClassifier,
    RandomForestClassifier, GradientBoostingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(n_estimators=100, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
lr = LogisticRegression(max_iter=1000, random_state=42)
est = [('rf', rf), ('gb', gb), ('lr', lr)]

results = {}
for n, m in est:
    c = m.__class__(**m.get_params()).fit(X_tr, y_tr)
    results[n] = accuracy_score(y_te, c.predict(X_te))

vc = VotingClassifier(estimators=est, voting='soft').fit(X_tr, y_tr)
results['voting'] = accuracy_score(y_te, vc.predict(X_te))

sc = StackingClassifier(estimators=est, final_estimator=LogisticRegression(max_iter=1000), cv=5).fit(X_tr, y_tr)
results['stacking'] = accuracy_score(y_te, sc.predict(X_te))

for n, a in sorted(results.items(), key=lambda x: x[1]):
    print(f'  {n:<10} {a:.4f}{" ← best" if a == max(results.values()) else ""}')

coefs = sc.final_estimator_.coef_[0]
for n, c in zip(['rf','gb','lr'], coefs):
    print(f'  {n} weight: {c:.3f}')

</details>

---

## Exercise 5 (Hard): LLM Router Simulator

**📖 Concept:** Cascade: cheap → medium → expensive. Route 80% to cheap → 70%+ savings.

**📝 Task:** 3 models with different cost. Cascade router. Cost analysis vs always-expensive.

In [ ]:
# ✏️ YOUR CODE HERE
from sklearn.tree import DecisionTreeClassifier

costs = {'cheap': 0.075, 'medium': 0.50, 'expensive': 3.00}
# TODO: cascade router, cost analysis

<details><summary>✅ Solution</summary>



In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score

costs = {'cheap': 0.075, 'medium': 0.50, 'expensive': 3.00}
models = {
    'cheap': DecisionTreeClassifier(max_depth=5, random_state=42),
    'medium': RandomForestClassifier(n_estimators=50, random_state=42),
    'expensive': GradientBoostingClassifier(n_estimators=200, random_state=42),
}
for n, m in models.items():
    m.fit(X_tr, y_tr)
    print(f'  {n:>10}: acc={accuracy_score(y_te, m.predict(X_te)):.4f}, ${costs[n]}/1K')

preds = {n: m.predict(X_te) for n, m in models.items()}
exp = preds['expensive']

routed = {'cheap': 0, 'medium': 0, 'expensive': 0}
total_cost = 0
for i in range(len(X_te)):
    if preds['cheap'][i] == exp[i]:
        routed['cheap'] += 1; total_cost += costs['cheap']/1000
    elif preds['medium'][i] == exp[i]:
        routed['medium'] += 1; total_cost += costs['medium']/1000
    else:
        routed['expensive'] += 1; total_cost += costs['expensive']/1000

n = len(X_te)
expensive_cost = n * costs['expensive'] / 1000
print(f'\nRouting: {routed}')
print(f'Cost: ${total_cost:.3f} vs ${expensive_cost:.3f} (always-expensive)')
print(f'Savings: {(1-total_cost/expensive_cost)*100:.1f}%')

</details>

---

## Exercise 6 (Hard): Full Ensemble Pipeline

**📖 Concept:** All methods: bagging, boosting, voting, stacking, cascade. Compare accuracy + time + cost.

**📝 Task:** Train all 5, unified report, recommend best for production.

In [ ]:
# ✏️ YOUR CODE HERE
# TODO: all 5 ensemble methods, report, recommend

<details><summary>✅ Solution</summary>



In [ ]:
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
    VotingClassifier, StackingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import time

rf = RandomForestClassifier(n_estimators=100, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
lr = LogisticRegression(max_iter=1000, random_state=42)
est = [('rf', rf), ('gb', gb), ('lr', lr)]

R = {}
t0=time.time(); RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
rf2 = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
R['RF (bagging)'] = (accuracy_score(y_te, rf2.predict(X_te)), time.time()-t0, 0.50)

t0=time.time(); gb2 = GradientBoostingClassifier(n_estimators=200, random_state=42).fit(X_tr, y_tr)
R['GB (boost)'] = (accuracy_score(y_te, gb2.predict(X_te)), time.time()-t0, 3.00)

t0=time.time(); vc = VotingClassifier(estimators=est, voting='soft').fit(X_tr, y_tr)
R['Voting'] = (accuracy_score(y_te, vc.predict(X_te)), time.time()-t0, 3.58)

t0=time.time(); sc = StackingClassifier(estimators=est, final_estimator=LogisticRegression(max_iter=1000), cv=5).fit(X_tr, y_tr)
R['Stacking'] = (accuracy_score(y_te, sc.predict(X_te)), time.time()-t0, 3.58)

# Cascade
dt = DecisionTreeClassifier(max_depth=5, random_state=42).fit(X_tr, y_tr)
p_c, p_m, p_e = dt.predict(X_te), rf2.predict(X_te), gb2.predict(X_te)
cost = sum(0.075/1000 if p_c[i]==p_e[i] else (0.5/1000 if p_m[i]==p_e[i] else 3.0/1000) for i in range(len(X_te)))
R['Cascade'] = (accuracy_score(y_te, p_e), 0.01, cost/len(X_te)*1000)

print('═'*50)
print('  ENSEMBLE PIPELINE REPORT')
print('═'*50)
print(f'\n{"Method":<14} {"Acc":>6} {"Time":>7} {"$/1K":>7}')
print('-'*38)
for n, (a, t, c) in R.items():
    print(f'  {n:<12} {a:.4f} {t:>6.2f}s ${c:>5.2f}')
print(f'\n  Best cost/acc: Cascade Router')

</details>

---

## 🎉 Done!

All ensemble patterns mastered: bagging, boosting, voting, stacking, cascade routing.

Cascade routing IS how production LLM systems save 70%+ on costs. **Module 7** and **Module 12** build on these exact patterns.